In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_excel('test_dataset_anonymized.xlsx')
df.head()  # view first 5 rows

,user_id,is_trial,first_event_date,subscription_renewal_amount,Unnamed: 4,Subs type,Trial period,Subscription period,First payment,Rebill payment
0,21889,1,2022-01-01,0,NaN,trial,7 days,1 month,6.99,29.99
1,21890,1,2019-07-01,0,NaN,wo trial,No trial,3 months,40.00,40.00
2,21891,1,2019-07-01,0,NaN,Subscription is renewed automatically\t\t\t,NaN,NaN,NaN,NaN
3,21892,1,2019-07-01,0,NaN,NaN,NaN,NaN,NaN,NaN
4,21894,1,2019-07-01,0,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
# Let's keep only the columns we need to work with:
df = df[['user_id','is_trial','first_event_date','subscription_renewal_amount']]

# Check the data types:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9426 entries, 0 to 9425
Data columns (total 4 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   user_id                      9426 non-null   int64         
 1   is_trial                     9426 non-null   int64         
 2   first_event_date             9426 non-null   datetime64[us]
 3   subscription_renewal_amount  9426 non-null   int64         
dtypes: datetime64[us](1), int64(3)
memory usage: 294.7 KB


In [32]:
# check dataframe size:
df.shape

# explore basic stats:
df.describe()


,user_id,is_trial,first_event_date,subscription_renewal_amount
count,9426.000000,9426.000000,9426,9426.000000
mean,23609.311161,0.338320,2019-07-05 04:05:57.479312,5.569489
min,21889.000000,0.000000,2019-07-01 00:00:00,0.000000
25%,22736.000000,0.000000,2019-07-03 00:00:00,0.000000
50%,23572.000000,0.000000,2019-07-06 00:00:00,2.000000
75%,24428.000000,1.000000,2019-07-07 00:00:00,9.000000
max,115217.000000,1.000000,2022-01-01 00:00:00,26.000000
std,1779.225697,0.473163,NaN,6.964244


1: Finding total revenue for trial users: 


In [33]:
# filter trial users
trial_df = df[df['is_trial'] == 0]
trial_df.head()

# calculate revenue per user
trial_df['revenue'] = 6.99 + trial_df['subscription_renewal_amount'] * 29.99

# total revenue
total_revenue_trial = trial_df['revenue'].sum()

print(f'Total revenue for trial users: ${total_revenue_trial:,.2f}')

Total revenue for trial users: $1,617,981.66


2a: Segmenting users by number of renewals:

In [34]:
# Number of users by renewal count:
renewals_by_users = df.groupby('subscription_renewal_amount')['user_id'].count().reset_index().rename(columns={'user_id': 'number_of_users',
                                                                                                    'subscription_renewal_amount': 'number_of_renewals'})

print(renewals_by_users)

    number_of_renewals  number_of_users
0                    0             3188
1                    1              982
2                    2              633
3                    3              498
4                    4              411
5                    5              353
6                    6              321
7                    7              281
8                    8              252
9                    9              236
10                  10              228
11                  11              206
12                  12              190
13                  13              174
14                  14              163
15                  15              147
16                  16              139
17                  17              134
18                  18              126
19                  19              117
20                  20              108
21                  21              102
22                  22               96
23                  23               92


2b: Total Revenue by segments (for Trial only): 

In [35]:
revenue_by_renewals = trial_df.groupby('subscription_renewal_amount')['revenue'].sum().reset_index().rename(columns={'subscription_renewal_amount': 'number_of_renewals', 'revenue': 'revenue_by_renewals'}).sort_values('number_of_renewals')

print(revenue_by_renewals)

    number_of_renewals  revenue_by_renewals
0                    1             36277.38
1                    2             42392.01
2                    3             48286.08
3                    4             52176.45
4                    5             55399.82
5                    6             60004.53
6                    7             60954.52
7                    8             62221.32
8                    9             65348.40
9                   10             69970.92
10                  11             69397.28
11                  12             69705.30
12                  13             69053.64
13                  14             69576.55
14                  15             67155.48
15                  16             67669.37
16                  17             69253.88
17                  18             68898.06
18                  19             67485.60
19                  20             65533.32
20                  21             64951.56
21                  22          

3: Average LTV by subscription types

In [36]:
# Calculate LTV per user for both subscription types
df['ltv'] = np.where(
    df['is_trial'] == 0,
    6.99 + df['subscription_renewal_amount'] * 29.99,
    40.00 + df['subscription_renewal_amount'] * 40.00
)

# Average LTV by subsription type
ltv_by_type = df.groupby('is_trial')['ltv'].agg(['mean', 'median']).reset_index().rename(columns={'ltv': 'avg_ltv'})

print(ltv_by_type)

   is_trial        mean  median
0         0  259.416652  186.93
1         1   40.012543   40.00


4: Quarterly revenue

In [37]:
# Create payment timeline for each user
df['number_of_renewals'] = df['subscription_renewal_amount'].apply(
    lambda x: list(range(0, x + 1))
) 

# With 'explode' - unpack each user row into one row per payment
payments_df = df.explode('number_of_renewals').reset_index(drop=True)

payments_df.head(10)

,user_id,is_trial,first_event_date,subscription_renewal_amount,ltv,number_of_renewals
0,21889,1,2022-01-01,0,40.0,0
1,21890,1,2019-07-01,0,40.0,0
2,21891,1,2019-07-01,0,40.0,0
3,21892,1,2019-07-01,0,40.0,0
4,21894,1,2019-07-01,0,40.0,0
5,21895,1,2019-07-01,0,40.0,0
6,21896,1,2019-07-01,0,40.0,0
7,21897,1,2019-07-01,0,40.0,0
8,21898,1,2019-07-01,0,40.0,0
9,21899,1,2019-07-01,0,40.0,0


In [38]:
# Calculate payment date for each renewal
payments_df['payment_date'] = np.where(
    payments_df['is_trial'] == 0,
    payments_df['first_event_date'] + pd.to_timedelta(7 + payments_df['number_of_renewals'] * 30, unit='D'),
    payments_df['first_event_date'] + pd.to_timedelta(payments_df['number_of_renewals'] * 90, unit='D')
)

# Calculate payment amount for each renewal
payments_df['payment_amount'] = np.where(
    payments_df['is_trial'] == 0,
    np.where(payments_df['number_of_renewals'] == 0, 6.99, 29.99),
    40.00
)

payments_df.head(9000)

,user_id,is_trial,first_event_date,subscription_renewal_amount,ltv,number_of_renewals,payment_date,payment_amount
0,21889,1,2022-01-01,0,40.00,0,2022-01-01,40.00
1,21890,1,2019-07-01,0,40.00,0,2019-07-01,40.00
2,21891,1,2019-07-01,0,40.00,0,2019-07-01,40.00
3,21892,1,2019-07-01,0,40.00,0,2019-07-01,40.00
4,21894,1,2019-07-01,0,40.00,0,2019-07-01,40.00
...,...,...,...,...,...,...,...,...
8995,23213,0,2019-07-05,4,126.95,2,2019-09-10,29.99
8996,23213,0,2019-07-05,4,126.95,3,2019-10-10,29.99
8997,23213,0,2019-07-05,4,126.95,4,2019-11-09,29.99
8998,23216,0,2019-07-05,4,126.95,0,2019-07-12,6.99


In [ ]:
# Self-check that 'explode' worked out correctly
print(payments_df.shape)
print(payments_df['number_of_renewals'].unique())

(61924, 8)
[0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26]


In [41]:
# Extract quarter from payment date
payments_df['quarter'] = payments_df['payment_date'].dt.to_period('Q')

# Group by quarter and sum revenue
quartely_revenue = payments_df.groupby('quarter')['payment_amount'].sum().reset_index().rename(columns={'payment_amount': 'total_revenue'}).sort_values('quarter')

print(quartely_revenue)

  quarter  total_revenue
0  2019Q3      515791.70
1  2019Q4      373775.38
2  2020Q1      274708.40
3  2020Q2      204561.79
4  2020Q3      151029.64
5  2020Q4      109793.39
6  2021Q1       66307.89
7  2021Q2       37157.61
8  2021Q3       12415.86
9  2022Q1          40.00
